# ASR 모델 테스트 — Apple Silicon (macOS M4 Pro)

`lab/ASR-model-test/test-way.md`(ASR 평가 프로토콜)과 `ASR-test-lab.md`를 따릅니다.

Colab/T4 전제 엔진(faster-whisper 등) 대신 **Apple Silicon 최적화 프레임워크**(`mlx-whisper`, torch MPS)로 교체했고,
엔진 교체는 **어댑터 패턴**(test-way.md §3.10)으로 구현되어 있습니다.

- **테스트 트랙** (test-way.md §4.1): clean / noisy(SNR 5dB) / code-switched / silence / streaming
- **지표 모듈**: `asr_metrics.py` (CER·WER, RTF·지연, 신뢰도·환각, 노이즈 주입, 벤치마크 하네스)
- **어댑터 모듈**: `asr_adapters.py` (계약 dict + 팩토리)
- **유틸 모듈**: `asr_utils.py` (환경 체크, 오디오 I/O, TTS 평가 세트, 스트리밍 시뮬레이션)

## 0. 환경 확인

In [1]:
import json
import logging
import os
import sys
from pathlib import Path

logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("urllib3").setLevel(logging.ERROR)

# ── import 경로 부트스트랩 ──────────────────────────────────────
NB_DIR = Path.cwd()
if (NB_DIR / "asr_adapters.py").exists():
    sys.path.insert(0, str(NB_DIR))
else:
    sys.path.insert(0, str(NB_DIR / "lab" / "ASR-model-test"))
REPO_ROOT = NB_DIR if (NB_DIR / "app.py").exists() else NB_DIR.parent.parent

import numpy as np
import pandas as pd

from asr_utils import (env_check_mac, build_eval_set, summarize_eval_set,
                       load_audio, simulate_stream, save_results_json,
                       DATASETS, SAMPLE_RATE)
from asr_metrics import (cer, speech_ratio, is_hallucination,
                         hallucination_rate, streaming_metrics,
                         run_benchmark, quantization_gap)
from asr_adapters import (create_adapter, adapter_available,
                          make_transcribe_fn)

env_check_mac()
print(f"\n샘플레이트: {SAMPLE_RATE} Hz")

 ✅ 파이썬 3.10+            3.14.6
 ✅ PyTorch (MPS)        Apple Silicon (MPS)
 ✅ Apple MLX            Apple native ML
 ✅ 엔진 mlx-whisper       Apple Silicon 네이티브 Whisper
 ✅ 엔진 openai-whisper    torch MPS 기준선
 ✅ 엔진 sensevoice        SenseVoice-Small (CTC)
 ❌ 엔진 qwen3-asr         Qwen3-ASR-0.6B (LLM)
 ✅ 패키지 gtts             
 ✅ 패키지 librosa          
 ✅ 패키지 soundfile        
 ✅ 패키지 jiwer            
 ✅ 패키지 numpy            
 ✅ 패키지 matplotlib       
 ✅ 패키지 koreanize_matplotlib 
 ✅ 패키지 seaborn          
 ✅ 패키지 pandas           
--------------------------------------------------------
 통과 15/16 → ❌ 미설치 항목 조치 후 재실행

샘플레이트: 16000 Hz


## 1. 테스트 트랙 데이터 생성 (gTTS)

clean · code-switched 는 gTTS(한국어 TTS)로 합성하고, noisy 는 clean 에 SNR 5dB 백색소음을 주입합니다
(`asr_metrics.add_noise`). silence 는 순수 침묵(0dBFS)과 미세 소음(≈-60dBFS) 2종입니다.
생성된 wav는 `data/temp_audio/`(gitignore 대상)에 저장됩니다.

In [2]:
DATASET_ID = "v1"   # ← 두 번째 실험 사이클에서 "v2" 로 변경
DATA_DIR = REPO_ROOT / "data" / "temp_audio" / "asr_test" / DATASET_ID
RESULT_DIR = NB_DIR / "results"

eval_set = build_eval_set(DATASETS[DATASET_ID], DATA_DIR)
print(f"[{DATASET_ID}] 평가 세트 {len(eval_set)}건 → {DATA_DIR}")
summarize_eval_set(eval_set)

/opt/miniconda3/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[v1] 평가 세트 18건 → /Users/heewoo/_projects/lecture-note-ai/data/temp_audio/asr_test/v1
  clean          5건  길이 5.5s ~ 6.9s
  code_switched  4건  길이 5.6s ~ 7.5s
  noisy          5건  길이 5.5s ~ 6.9s
  silence        4건  길이 4.0s ~ 4.0s


## 2. 어댑터 팩토리 + 스모크 테스트

사용 가능한 엔진만 팩토리로 생성합니다. 미설치 엔진(sensevoice / qwen3-asr)은 건너뜁니다.
각 어댑터는 `transcribe(audio) -> 계약 dict` (test-way.md §3.10)를 반환합니다.

In [3]:
ENGINES = ["mlx-whisper", "openai-whisper", "sensevoice", "qwen3-asr"]
ENABLED = {e: adapter_available(e) for e in ENGINES}
print("엔진 가용성:", "  ".join(f"{k} {'✅' if v else '❌'}" for k, v in ENABLED.items()))

adapters = {e: create_adapter(e) for e in ENGINES if ENABLED[e]}
print("\n[스모크 테스트] clean 1건")
for e, a in adapters.items():
    r = a.transcribe(eval_set[0][0], utt_id="smoke")
    print(f"  [{e}] {r['text'][:38]!r}  conf={r['confidence_ok']}  "
          f"avg_logprob={r['avg_logprob']}")

Notice: ffmpeg is not installed. torchaudio is used to load audio
If you want to use ffmpeg backend to load audio, please install it by:
	sudo apt install ffmpeg # ubuntu
	# brew install ffmpeg # mac


엔진 가용성: mlx-whisper ✅  openai-whisper ✅  sensevoice ✅  qwen3-asr ❌

[스모크 테스트] clean 1건


2026-07-31 23:45:20,452 [INFO] HTTP Request: GET https://huggingface.co/api/models/mlx-community/whisper-large-v3-turbo/revision/main "HTTP/1.1 200 OK"


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 1697.07it/s]

...

2026-07-31 23:45:25,634 [INFO] ckpt: /Users/heewoo/.cache/huggingface/hub/models--FunAudioLLM--SenseVoiceSmall/snapshots/3847d57b6bdf2dd8875cb1508d2af43d80a16bf7/model.pt


2026-07-31 23:45:25,832 [INFO] scope_map: ['module.', 'None']


2026-07-31 23:45:25,832 [INFO] excludes: None


2026-07-31 23:45:25,862 [INFO] Loading ckpt: /Users/heewoo/.cache/huggingface/hub/models--FunAudioLLM--SenseVoiceSmall/snapshots/3847d57b6bdf2dd8875cb1508d2af43d80a16bf7/model.pt, status: <All keys matched successfully>


  [sensevoice] '오늘 강의 에 서는 데이터베이스 트렌 젝션 의 4 가지 성질 에 대해'  conf=True  avg_logprob=None


## 3. 비교 매트릭스 (test-way.md §4.3)

각 엔진을 `run_benchmark` 하네스에 태워 조건별 CER · 평균 지연 · RTF를 계산합니다.

In [4]:
matrix_rows = []
for e, a in adapters.items():
    row = run_benchmark(e, make_transcribe_fn(a), eval_set)
    row.pop("n", None)
    matrix_rows.append(row)

matrix_df = pd.DataFrame(matrix_rows).set_index("engine")
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
matrix_df

  0%|          | 0/624 [00:00<?, ?frames/s]

100%|██████████| 624/624 [00:00<00:00, 1345.69frames/s]

100%|██████████| 624/624 [00:00<00:00, 1344.11frames/s]

  0%|          | 0/691 [00:00<?, ?frames/s]

100%|██████████| 691/691 [00:00<00:00, 1528.21frames/s]

...

100%|██████████| 566/566 [00:00<00:00, 1319.20frames/s]

[mlx-whisper] clean CER 0.7% | code_switched CER 55.3% | noisy CER 3.5% | silence CER 100.0% | 평균 지연 439ms | RTF 0.08


[openai-whisper] clean CER 0.7% | code_switched CER 55.3% | noisy CER 3.5% | silence CER 100.0% | 평균 지연 677ms | RTF 0.12


[sensevoice] clean CER 8.5% | code_switched CER 64.3% | noisy CER 13.5% | silence CER 100.0% | 평균 지연 541ms | RTF 0.10


,cer_clean,cer_code_switched,cer_noisy,cer_silence,latency_avg_ms,rtf_avg
engine,,,,,,
mlx-whisper,0.007,0.553,0.035,1.000,439.066,0.079
openai-whisper,0.007,0.553,0.035,1.000,676.607,0.119
sensevoice,0.085,0.643,0.135,1.000,540.791,0.098


## 4. 침묵 트랙 → 환각 방어 (P1)

모든 엔진이 침묵에서 환각을 낼 수 있으므로(test-way.md §3.2), VAD 게이트에 더해 텍스트 레벨 3신호 필터를 적용합니다.
아래는 엔진별 환각률(`hallucination_rate`)과 실제 환각 텍스트 예시입니다.

In [5]:
def silence_analysis(adapter):
    recs = []
    for path, ref, cond, dur in eval_set:
        if cond != "silence":
            continue
        y = load_audio(path)
        r = adapter.transcribe(path, utt_id=Path(path).stem)
        ratio = speech_ratio(y, SAMPLE_RATE)
        recs.append({"utt_id": r["utt_id"], "text": r["text"],
                     "speech_ratio": round(ratio, 3),
                     "hallucination": is_hallucination(r["text"], dur, ratio)})
    return recs

hallu_rows = []
for e, a in adapters.items():
    recs = silence_analysis(a)
    rate = hallucination_rate([r["text"] for r in recs],
                             [r["speech_ratio"] for r in recs])
    hallu_rows.append({"engine": e, "hallucination_rate": rate})
    for r in recs:
        r["engine"] = e
    print(f"[{e}] 환각률 {rate:.0%}",
          " | 환각 텍스트:", [r['text'] for r in recs if r['hallucination']] or "없음")

hallucination_df = pd.DataFrame(hallu_rows).set_index("engine")
hallucination_df

  0%|          | 0/400 [00:00<?, ?frames/s]

100%|██████████| 400/400 [00:00<00:00, 988.22frames/s]

100%|██████████| 400/400 [00:00<00:00, 986.41frames/s]

  0%|          | 0/400 [00:00<?, ?frames/s]

100%|██████████| 400/400 [00:00<00:00, 1031.53frames/s]

...

100%|██████████| 400/400 [00:00<00:00, 1037.13frames/s]

[mlx-whisper] 환각률 100%  | 환각 텍스트: ['감사합니다.', '감사합니다.', '감사합니다.', '감사합니다.']


[openai-whisper] 환각률 100%  | 환각 텍스트: ['다음 영상에서 만나요.', '다음 영상에서 만나요.', '감사합니다.', '감사합니다.']


[sensevoice] 환각률 100%  | 환각 텍스트: ['그.', '그.', '그.', '그.']


,hallucination_rate
engine,
mlx-whisper,1.000
openai-whisper,1.000
sensevoice,1.000


## 5. 2-Tier 신뢰도 전략 (P2)

greedy 디코딩 → `avg_logprob < -1.0` 이면 beam=5 로 1회 재시도 (test-way.md §3.3).

In [6]:
if "mlx-whisper" in adapters:
    mlx = adapters["mlx-whisper"]
    noisy_items = [x for x in eval_set if x[2] == "noisy"]

    # 신뢰도가 가장 낮은 noisy 발화 선택
    worst_item, worst_lp = None, float("inf")
    for it in noisy_items:
        lp = mlx.transcribe(it[0])["avg_logprob"] or -99.0
        if lp < worst_lp:
            worst_item, worst_lp = it, lp

    print(f"최저 신뢰도 noisy 발화: {Path(worst_item[0]).name}  avg_logprob={worst_lp:.3f}")
    greedy = mlx.transcribe(worst_item[0], utt_id="tier1_greedy")
    print(f"  greedy : {greedy['text']}  conf={greedy['confidence_ok']}")

    retry = greedy
    if not greedy["confidence_ok"]:
        mlx_beam = create_adapter("mlx-whisper", beam_size=5)
        retry = mlx_beam.transcribe(worst_item[0], utt_id="tier2_beam")
        print(f"  beam=5 : {retry['text']}  conf={retry['confidence_ok']}")

    print(f"  CER  greedy={cer(worst_item[1], greedy['text']):.1%} → beam={cer(worst_item[1], retry['text']):.1%}")
else:
    print("mlx-whisper 없음 — 건너뜀")

  0%|          | 0/624 [00:00<?, ?frames/s]

100%|██████████| 624/624 [00:00<00:00, 1328.15frames/s]

100%|██████████| 624/624 [00:00<00:00, 1326.46frames/s]

  0%|          | 0/691 [00:00<?, ?frames/s]

100%|██████████| 691/691 [00:00<00:00, 1517.50frames/s]

...

최저 신뢰도 noisy 발화: utt_003_noisy5.wav  avg_logprob=-0.233


  0%|          | 0/667 [00:00<?, ?frames/s]

100%|██████████| 667/667 [00:00<00:00, 1491.35frames/s]

100%|██████████| 667/667 [00:00<00:00, 1489.83frames/s]

  greedy : 합성곡 신경망에서 킬링 레이어는 특징맵의 크기를 줄이는 역할을 합니다.  conf=True
  CER  greedy=6.7% → beam=6.7%


## 6. 양자화 허용 오차 (test-way.md §3.6)

`mlx-community/whisper-large-v3-turbo-q4`(4bit)와 fp16의 CER 차이를 **측정**합니다.
"측정하라, 가정하지 마라" — test-way.md §3.6.

In [7]:
if "mlx-whisper" in adapters:
    q4 = create_adapter("mlx-whisper",
                        model_id="mlx-community/whisper-large-v3-turbo-q4")
    items = [x for x in eval_set if x[2] in ("clean", "noisy")]
    # ModelHolder는 모델 1개만 캐시하므로, 모델별로 묶어 전사해
    # fp16↔q4 재로드를 최소화한다.
    hyps = {"fp16": {}, "q4": {}}
    for key, ad in (("fp16", mlx), ("q4", q4)):
        for path, ref, cond, dur in items:
            hyps[key][Path(path).stem] = ad.transcribe(path)["text"]

    quant_rows = []
    print(f"{'item':<16}{'cond':<8}{'fp16 CER':>10}{'q4 CER':>10}{'Δ':>10}")
    for path, ref, cond, dur in items:
        stem = Path(path).stem
        c_fp = cer(ref, hyps["fp16"][stem])
        c_q4 = cer(ref, hyps["q4"][stem])
        gap = quantization_gap(c_fp, c_q4)
        quant_rows.append({"item": stem, "cond": cond,
                           "fp16_cer": c_fp, "q4_cer": c_q4,
                           "gap_abs": gap["absolute"]})
        print(f"{stem:<16}{cond:<8}{c_fp:>10.1%}{c_q4:>10.1%}{gap['absolute']:>10.1%}")
    quant_df = pd.DataFrame(quant_rows)
    print("\n평균 CER 차이(q4−fp16):",
          f"{quant_df['gap_abs'].mean():+.2%}")

  0%|          | 0/624 [00:00<?, ?frames/s]

100%|██████████| 624/624 [00:00<00:00, 1383.21frames/s]

100%|██████████| 624/624 [00:00<00:00, 1381.87frames/s]

  0%|          | 0/691 [00:00<?, ?frames/s]

100%|██████████| 691/691 [00:00<00:00, 1528.29frames/s]

...

100%|██████████| 552/552 [00:00<00:00, 1273.27frames/s]

  0%|          | 0/566 [00:00<?, ?frames/s]

100%|██████████| 566/566 [00:00<00:00, 1329.32frames/s]

100%|██████████| 566/566 [00:00<00:00, 1327.80frames/s]

item            cond      fp16 CER    q4 CER         Δ
utt_001         clean         3.3%      3.3%      0.0%
utt_002         clean         0.0%      0.0%      0.0%
utt_003         clean         0.0%      0.0%      0.0%
utt_004         clean         0.0%      0.0%      0.0%
utt_005         clean         0.0%      0.0%      0.0%
utt_001_noisy5  noisy         3.3%      3.3%      0.0%
utt_002_noisy5  noisy         3.6%      0.0%     -3.6%
utt_003_noisy5  noisy         6.7%     10.0%      3.3%
utt_004_noisy5  noisy         0.0%      0.0%      0.0%
utt_005_noisy5  noisy         3.8%      3.8%      0.0%

평균 CER 차이(q4−fp16): -0.02%


## 7. initial_prompt 도메인 주입 (P5 대안)

한국어 파인튜닝 대신 무비용으로 도메인 컨텍스트를 주입해 코드 스위칭 발화의 CER을 개선합니다
(test-way.md §3.5, §3.9).

In [8]:
if "mlx-whisper" in adapters:
    prompt = "API, response, timeout, tokenizer, embedding, gradient, memory, database"
    prompted = create_adapter("mlx-whisper", initial_prompt=prompt)
    cs_items = [x for x in eval_set if x[2] == "code_switched"]
    print(f"initial_prompt: {prompt!r}\n")
    print(f"{'item':<14}{'base CER':>10}{'prompt CER':>12}")
    prompt_rows = []
    for path, ref, cond, dur in cs_items:
        t_base = mlx.transcribe(path)["text"]
        t_prompt = prompted.transcribe(path)["text"]
        c_base, c_prompt = cer(ref, t_base), cer(ref, t_prompt)
        prompt_rows.append({"item": Path(path).stem, "base_cer": c_base,
                           "prompt_cer": c_prompt})
        print(f"{Path(path).stem:<14}{c_base:>10.1%}{c_prompt:>12.1%}")
    prompt_df = pd.DataFrame(prompt_rows)

initial_prompt: 'API, response, timeout, tokenizer, embedding, gradient, memory, database'

item            base CER  prompt CER


2026-07-31 23:46:14,223 [INFO] HTTP Request: GET https://huggingface.co/api/models/mlx-community/whisper-large-v3-turbo/revision/main "HTTP/1.1 200 OK"


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 2022.33it/s]

  0%|          | 0/652 [00:00<?, ?frames/s]

...

100%|██████████| 748/748 [00:00<00:00, 1597.42frames/s]

  0%|          | 0/748 [00:00<?, ?frames/s]

100%|██████████| 748/748 [00:00<00:00, 1743.27frames/s]

100%|██████████| 748/748 [00:00<00:00, 1741.06frames/s]

utt_cs_004         76.8%        0.0%


## 8. 스트리밍 시뮬레이션 (P6)

성장 버퍼 재전사 방식으로 실시간 스트리밍을 시뮬레이션하고
`first_partial_ms` / `final_latency_ms` / RTF를 측정합니다 (test-way.md §2.2, §3.8).

In [9]:
if "mlx-whisper" in adapters:
    clean_item = next(x for x in eval_set if x[2] == "clean")
    y = load_audio(clean_item[0])
    events, speech_ms, wall_s = simulate_stream(y, mlx, chunk_s=1.0,
                                                eou_silence_chunks=2)
    stream_metrics = streaming_metrics(events, speech_ms, wall_s)
    print(f"오디오 길이: {len(y)/SAMPLE_RATE:.1f}s  발화 판정: {speech_ms:.0f}ms  "
          f"벽시계: {wall_s:.1f}s")
    print(f"streaming_metrics: {stream_metrics}")
    display(pd.DataFrame(events))
else:
    print("mlx-whisper 없음 — 건너뜀")

  0%|          | 0/100 [00:00<?, ?frames/s]

100%|██████████| 100/100 [00:00<00:00, 252.44frames/s]

100%|██████████| 100/100 [00:00<00:00, 252.15frames/s]

  0%|          | 0/200 [00:00<?, ?frames/s]

100%|██████████| 200/200 [00:00<00:00, 490.93frames/s]

...

  0%|          | 0/624 [00:00<?, ?frames/s]

100%|██████████| 624/624 [00:00<00:00, 1325.57frames/s]

100%|██████████| 624/624 [00:00<00:00, 1323.88frames/s]

오디오 길이: 6.2s  발화 판정: 6240ms  벽시계: 3.1s
streaming_metrics: {'first_partial_ms': 1000.0, 'final_latency_ms': 0.0, 'rtf': 0.4969804286859541}


,event,t_ms,text
0,partial,1000.000,오늘 강의에서
1,partial,2000.000,오늘 강의에서는 데이터
2,partial,3000.000,오늘 강의에서는 데이터베이스 트랜
3,partial,4000.000,오늘 강의에서는 데이터베이스 트랜잭션의 4강의
4,partial,5000.000,오늘 강의에서는 데이터베이스 트랜잭션의 4가지 성질에 대해
5,partial,6000.000,오늘 강의에서는 데이터베이스 트랜잭션의 4가지 성질에 대해 다룹니다.
6,partial,6240.000,오늘 강의에서는 데이터베이스 트랜잭션의 4가지 성질에 대해 다룹니다.
7,final,6240.000,오늘 강의에서는 데이터베이스 트랜잭션의 4가지 성질에 대해 다룹니다.


## 9. 결과 저장

이번 사이클의 지표를 JSON으로 저장해 두 사이클 종합 결과(마크다운) 작성에 사용합니다.

In [10]:
report = {
    "dataset_id": DATASET_ID,
    "enabled_engines": list(adapters),
    "matrix": matrix_rows,
    "hallucination": hallu_rows,
    "streaming": stream_metrics if "stream_metrics" in dir() else None,
}
save_results_json(report, RESULT_DIR / f"asr_results_{DATASET_ID}.json")
print(f"저장 완료 → {RESULT_DIR / f'asr_results_{DATASET_ID}.json'}")

저장 완료 → /Users/heewoo/_projects/lecture-note-ai/lab/ASR-model-test/results/asr_results_v1.json
